# Ejercicio 5 - Pipeline de scikit-learn con MLflow

Este notebook entrena un clasificador Wine y registra el pipeline completo en MLflow. Usa **Experiments** para guardar parámetros y métricas, y **MLflow Models** para guardar el pipeline reproducible.

**Databricks:** importa este archivo `.ipynb`, adjuntalo a un cluster **Databricks Runtime ML** con MLflow 3 o superior y selecciona **Run all**. El notebook usa el tracking administrado de Databricks y el experimento asociado al notebook. No requiere configurar tokens, URL de tracking ni Feature Store.

In [ ]:
import os

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if int(mlflow.__version__.split(".")[0]) < 3:
    raise RuntimeError("Este ejercicio requiere MLflow 3 o superior. Usa un Databricks Runtime ML compatible.")

IN_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
if not IN_DATABRICKS:
    mlflow.set_experiment("ejercicio-5-sklearn-mlflow")
    tracking_target = "MLflow local"
else:
    # Databricks registra por defecto en el experimento asociado a este notebook.
    tracking_target = "tracking administrado de Databricks"

print(f"MLflow {mlflow.__version__} | Tracking: {tracking_target}")

## 1. Preparar los datos y el pipeline

El dataset Wine viene incluido en scikit-learn, asi que no necesita archivos locales ni descargas. El preprocesamiento y el clasificador quedan dentro de un solo `Pipeline`.

In [ ]:
wine = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    wine.data,
    wine.target,
    test_size=0.25,
    random_state=42,
    stratify=wine.target,
)

params = {"n_estimators": 150, "max_depth": 5, "random_state": 42}
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(**params)),
    ]
)

print(f"Entrenamiento: {len(X_train)} filas | Prueba: {len(X_test)} filas | Features: {X_train.shape[1]}")

## 2. Entrenar y registrar en MLflow

Cada ejecución guarda los hiperparámetros, las métricas, las etiquetas del ejercicio y el pipeline con una firma de entrada/salida. Al cambiar hiperparámetros y volver a ejecutar, cada intento queda como un run separado para compararlo en Experiments.

In [ ]:
with mlflow.start_run(run_name="random-forest-wine-notebook"):
    mlflow.log_params({f"rf_{name}": value for name, value in params.items()})
    mlflow.log_param("test_size", 0.25)
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "f1_weighted": f1_score(y_test, predictions, average="weighted"),
    }
    mlflow.log_metrics(metrics)
    mlflow.set_tags({"exercise": "5", "framework": "scikit-learn", "dataset": "wine"})

    input_example = X_train.head(5)
    signature = infer_signature(X_train, pipeline.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="wine_random_forest_pipeline",
        signature=signature,
        input_example=input_example,
    )

print("Métricas:", {name: round(value, 4) for name, value in metrics.items()})
print("MLflow Model guardado en:", model_info.model_uri)